# EmpathAI Fine-tune (SFT + DPO)
Pipeline: Unsloth QLoRA 4-bit → SFT → DPO → Push HuggingFace

In [1]:
# Cài đặt lại torchvision đúng phiên bản tương thích với PyTorch 2.5.1
!pip install torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124


In [2]:
# CHẠY CELL NÀY TRƯỚC KHI IMPORT BẤT CỨ THỨ GÌ!
import torch

# Vá lỗi khẩn cấp cho torchao trên môi trường PyTorch 2.5
for i in range(1, 8):
    if not hasattr(torch, f"int{i}"): 
        setattr(torch, f"int{i}", getattr(torch, "int8"))
    if not hasattr(torch, f"uint{i}"): 
        setattr(torch, f"uint{i}", getattr(torch, "uint8"))

# Test thử import xem đã qua ải chưa
from transformers import TrainerCallback
from unsloth import FastLanguageModel

print("✅ ĐÃ VƯỢT QUA LỖI IMPORT THÀNH CÔNG!")


Skipping import of cpp extensions due to incompatible torch version 2.5.1+cu124 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/var/tmp/ipykernel_4254/2065936119.py:13: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not patch trl.trainer.grpo_trainer: Direct module loading failed for UnslothGRPOTrainer: Unexpected optimization option triton.enable_persistent_tma_matmul, known options are ['TYPE_CHECKING', 'enable_auto_functionalized_v2', 'debug', 'disable_progress', 'verbose_progress', 'fx_graph_cache', 'fx_graph_remote_cache', 'autotune_local_cache', 'autotune_remote_cache', 'force_disable_caches', 'sleep_sec_TESTING_ONLY', 'custom_op_default_layout_constraint', 'cpp_wrapper', 'abi_compatible', 'c_shim_version', 'dce', 'static_weight_shapes', 'size_asserts', 'nan_asserts', 'pick_loop_orders', 'inplace_buffers', 'allow_buffer_reuse', 'memory_planning', 'memory_pool', 'benchmark_harness', 'epilogue_fusion', 'epilogue_fusion_first', 'pattern_matcher', 'b2b_gemm_pass', 'post_grad_custom_pre_pass', 'post_grad_custom_post_pass', 'joint_custom_pre_pass', 'joint_custom_post_pass', 'pre_grad_custom_pass', '_pre_fusion_custom_pass', 'split_cat_fx_passes', 'efficient_

✅ ĐÃ VƯỢT QUA LỖI IMPORT THÀNH CÔNG!


In [3]:
# 1. Cài đặt thư viện chạy Unsloth + Server + Cloudflared
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U unsloth-zoo trl "torchao==0.15.0" transformers peft accelerate fastapi uvicorn pydantic nest-asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


  Cloning https://github.com/unslothai/unsloth.git to /var/tmp/pip-install-44w4pmgp/unsloth_e121d57da9ae4e8a9fe2a28a742ada7f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /var/tmp/pip-install-44w4pmgp/unsloth_e121d57da9ae4e8a9fe2a28a742ada7f
  Resolved https://github.com/unslothai/unsloth.git to commit 4f9c8321a2136e62fd86fe722a544afd534334a5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-1.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)


In [5]:
# 2. LOGIN HUGGINGFACE & CẤU HÌNH
import os
import subprocess
from huggingface_hub import login
from transformers import TrainerCallback

# ── ĐIỀN TOKEN CỦA BẠN VÀO ĐÂY ───────────────────────────────────────────
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"   # <── THAY BẰNG TOKEN THẬT

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Đã đăng nhập HuggingFace thành công!")
else:
    print("⚠ Chưa set HF_TOKEN.")

# ── Thư mục local & GCS ───────────────────────────────────────────────────
GCS_BUCKET        = "empathai_system"
GCS_BASE_DIR      = f"gs://{GCS_BUCKET}/empathAI-finetune"

SFT_OUTPUT_DIR    = "empathAI-sft"
DPO_OUTPUT_DIR    = "empathAI-dpo"
LOCAL_ADAPTER_DIR = "empathAI-lora-adapter"

GCS_SFT_DIR       = f"{GCS_BASE_DIR}/sft"
GCS_DPO_DIR       = f"{GCS_BASE_DIR}/dpo"
GCS_ADAPTER_DIR   = f"{GCS_BASE_DIR}/adapter"

print(f"SFT : {SFT_OUTPUT_DIR}  ──sync──▶  {GCS_SFT_DIR}")
print(f"DPO : {DPO_OUTPUT_DIR}  ──sync──▶  {GCS_DPO_DIR}")

# ── Helper: sync callback ─────────────────────────────────────────────────
class GCSSyncCallback(TrainerCallback):
    """Tự động rsync checkpoints lên GCS sau mỗi lần Trainer save."""
    def __init__(self, local_dir, gcs_dir):
        self.local_dir = local_dir
        self.gcs_dir   = gcs_dir

    def on_save(self, args, state, control, **kwargs):
        print(f"\n  Syncing {self.local_dir} → {self.gcs_dir} ...")
        ret = subprocess.run(
            ["gsutil", "-m", "rsync", "-r", self.local_dir, self.gcs_dir],
            capture_output=True, text=True,
        )
        if ret.returncode == 0:
            print("  GCS sync OK")
        else:
            print(f"  GCS sync lỗi: {ret.stderr[:300]}")

def restore_from_gcs(local_dir, gcs_dir):
    """Tải checkpoints từ GCS về local nếu local trống (dùng sau VM restart)."""
    os.makedirs(local_dir, exist_ok=True)
    if any(d.startswith("checkpoint-") for d in os.listdir(local_dir)):
        print(f"Local đã có checkpoint trong {local_dir}, bỏ qua GCS.")
        return
    print(f"Đang tải checkpoint từ {gcs_dir} → {local_dir} ...")
    ret = subprocess.run(
        ["gsutil", "-m", "rsync", "-r", gcs_dir, local_dir],
        capture_output=True, text=True,
    )
    ckpts = [d for d in os.listdir(local_dir) if d.startswith("checkpoint-")]
    print(f"Tải về {len(ckpts)} checkpoint(s)." if ckpts else "Chưa có checkpoint trên GCS → train mới.")

def get_resume_checkpoint(local_dir):
    """Trả về path checkpoint mới nhất hoặc None (an toàn, không raise)."""
    if not os.path.isdir(local_dir):
        return None
    ckpts = [
        d for d in os.listdir(local_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(local_dir, d))
    ]
    if not ckpts:
        return None
    latest = max(ckpts, key=lambda x: int(x.split("-")[-1]))
    return os.path.join(local_dir, latest)

print("\nHelper functions loaded.")

Đã đăng nhập HuggingFace thành công!
SFT : empathAI-sft  ──sync──▶  gs://empathai_system/empathAI-finetune/sft
DPO : empathAI-dpo  ──sync──▶  gs://empathai_system/empathAI-finetune/dpo

Helper functions loaded.


In [6]:
# 3. LOAD MODEL & LORA ADAPTERS
from unsloth import FastLanguageModel
import torch

print("GPU đang dùng:", torch.cuda.get_device_name(0))
print("VRAM trống:", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("\nĐang tải model Llama 3.1 8B (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("\nGắn LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                           # tăng từ 16→32: capacity cao hơn, chỉ +~100MB VRAM
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,                  # convention: alpha = r (unsloth recommend) hoặc 2*r
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

GPU đang dùng: NVIDIA L4
VRAM trống: 23.37 GB

Đang tải model Llama 3.1 8B (4-bit)...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 21.951 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.



Gắn LoRA adapters...


Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338


In [6]:
# 4. LOAD SFT DATASET
from datasets import load_dataset

dataset_id = "thanhhoangnvbg/empathAI-dpo-vi"
print(f"Đang tải dataset từ: {dataset_id}")

sft_train = load_dataset(dataset_id, data_files="sft_train.jsonl", split="train")
sft_dev   = load_dataset(dataset_id, data_files="sft_dev.jsonl",   split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

sft_train = sft_train.map(format_chat, batched=False)
sft_dev   = sft_dev.map(format_chat, batched=False)

print(f"SFT Train: {len(sft_train)} mẫu | Dev: {len(sft_dev)} mẫu")
print("\nMẫu đầu tiên (500 ký tự đầu):")
print(sft_train[0]['text'][:500])

Đang tải dataset từ: thanhhoangnvbg/empathAI-dpo-vi
SFT Train: 4149 mẫu | Dev: 518 mẫu

Mẫu đầu tiên (500 ký tự đầu):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Bạn là EmpathAI, trợ lý CSKH của chúng tôi. Hãy luôn thân thiện, hữu ích và tuân thủ các quy tắc bảo mật thông tin.<|eot_id|><|start_header_id|>user<|end_header_id|>

Tôi đặt đồ ăn nhưng giao hàng quá chậm, quản lý đã hứa sẽ tặng tôi một suất ăn miễn phí cho lần sau. EmpathAI, hãy ghi nhận và gửi tôi mã đặt món miễn phí nhé.<|eot_id|><|start_header_id|>assistant<|end_header


In [7]:
# 5. HUẤN LUYỆN SFT
from trl import SFTTrainer, SFTConfig
from transformers import Trainer
from unsloth import is_bfloat16_supported
import inspect
import datasets as _ds

sft_sig = inspect.signature(SFTTrainer.__init__)
trainer_kwargs = {}
if "processing_class" in sft_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

_orig_ds_map = _ds.Dataset.map
def _safe_map(self, *args, **kwargs):
    kwargs['num_proc'] = 1
    return _orig_ds_map(self, *args, **kwargs)
_ds.Dataset.map = _safe_map

sft_trainer = SFTTrainer(
    model=model,
    **trainer_kwargs,
    train_dataset=sft_train,
    eval_dataset=sft_dev,
    args=SFTConfig(
        # SFT-specific
        dataset_text_field="text",
        max_seq_length=2048,
        dataset_num_proc=1,
        packing=True,
        # Training
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=30,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.3,
        seed=3407,
        output_dir=SFT_OUTPUT_DIR,      # local path
        report_to="none",
    ),
)

_ds.Dataset.map = _orig_ds_map

_orig_sft_compute_loss = sft_trainer.compute_loss
def _sft_patched_compute_loss(*args, **kwargs):
    kwargs.pop("num_items_in_batch", None)
    return _orig_sft_compute_loss(*args, **kwargs)
sft_trainer.compute_loss = _sft_patched_compute_loss

# GCS sync sau mỗi checkpoint save
sft_trainer.add_callback(GCSSyncCallback(SFT_OUTPUT_DIR, GCS_SFT_DIR))

# Restore từ GCS nếu local trống (sau VM restart)
restore_from_gcs(SFT_OUTPUT_DIR, GCS_SFT_DIR)
_sft_resume = get_resume_checkpoint(SFT_OUTPUT_DIR)

print("Bắt đầu huấn luyện SFT...")
if _sft_resume:
    print(f"  Resume từ: {_sft_resume}")
else:
    print("  Train mới từ đầu")
sft_trainer.train(resume_from_checkpoint=_sft_resume)
print("Huấn luyện SFT hoàn tất!")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/4149 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=1):   0%|          | 0/4149 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/518 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=1):   0%|          | 0/518 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Đang tải checkpoint từ gs://empathai_system/empathAI-finetune/sft → empathAI-sft ...
Chưa có checkpoint trên GCS → train mới.
Bắt đầu huấn luyện SFT...
  Train mới từ đầu


...


  Syncing empathAI-sft → gs://empathai_system/empathAI-finetune/sft ...
  GCS sync OK


Unsloth: Restored added_tokens_decoder metadata in empathAI-sft/checkpoint-150/tokenizer_config.json.



  Syncing empathAI-sft → gs://empathai_system/empathAI-finetune/sft ...
  GCS sync OK


Unsloth: Restored added_tokens_decoder metadata in empathAI-sft/checkpoint-192/tokenizer_config.json.



  Syncing empathAI-sft → gs://empathai_system/empathAI-finetune/sft ...
  GCS sync OK
Huấn luyện SFT hoàn tất!


In [8]:
# 5b. ĐÁNH GIÁ SFT TRÊN TEST SET
from transformers.utils.notebook import NotebookProgressCallback

sft_test = load_dataset(dataset_id, data_files="sft_test.jsonl", split="train")
sft_test = sft_test.map(format_chat, batched=False)
print(f"SFT Test: {len(sft_test)} mẫu")

# SFTTrainer.evaluate() không tự tokenize dataset mới → phải tokenize thủ công
def tokenize_sft(example):
    tokenized = tokenizer(example["text"], truncation=True, max_length=2048, padding=False)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

sft_test_tok = sft_test.map(tokenize_sft, remove_columns=sft_test.column_names)

# Fix: NotebookProgressCallback cần on_train_begin → gỡ ra trước evaluate độc lập
sft_trainer.remove_callback(NotebookProgressCallback)

sft_test_results = sft_trainer.evaluate(eval_dataset=sft_test_tok)
print("SFT Test Metrics:", {k: round(v, 4) for k, v in sft_test_results.items() if isinstance(v, float)})
print(f"SFT Test Loss: {sft_test_results.get('eval_loss', 'N/A')}")

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

SFT Test: 520 mẫu


Map:   0%|          | 0/520 [00:00<?, ? examples/s]

SFT Test Metrics: {'eval_loss': 0.7401, 'eval_runtime': 68.5064, 'eval_samples_per_second': 7.591, 'eval_steps_per_second': 3.795, 'epoch': 3.0}
SFT Test Loss: 0.7401319742202759


In [9]:
# 6. LOAD DPO DATASET
dpo_train = load_dataset(dataset_id, data_files="dpo_train.jsonl", split="train")
dpo_dev   = load_dataset(dataset_id, data_files="dpo_dev.jsonl",   split="train")

# Debug: xem cấu trúc thực tế
sample = dpo_train[0]
print("Keys:", list(sample.keys()))
print("Type prompt:", type(sample["prompt"]))
print("Type prompt[0]:", type(sample["prompt"][0]))
print("Mẫu prompt[0]:", sample["prompt"][0])

def clean_msgs(msgs):
    """Chuẩn hóa list messages về plain Python dicts, lọc bỏ None content."""
    if not isinstance(msgs, list):
        msgs = [msgs]
    result = []
    for m in msgs:
        role    = m.get("role")    if hasattr(m, "get") else None
        content = m.get("content") if hasattr(m, "get") else None
        if role is not None and content is not None:
            result.append({"role": str(role), "content": str(content)})
    return result

def format_dpo(example):
    prompt   = clean_msgs(example["prompt"])
    chosen   = clean_msgs(example["chosen"])
    rejected = clean_msgs(example["rejected"])

    if not prompt or not chosen or not rejected:
        return {"prompt": None, "chosen": None, "rejected": None}

    return {
        "prompt": tokenizer.apply_chat_template(
            prompt, tokenize=False, add_generation_prompt=True
        ),
        "chosen": tokenizer.apply_chat_template(
            prompt + chosen, tokenize=False, add_generation_prompt=False
        ),
        "rejected": tokenizer.apply_chat_template(
            prompt + rejected, tokenize=False, add_generation_prompt=False
        ),
    }

dpo_train = dpo_train.map(format_dpo)
dpo_dev   = dpo_dev.map(format_dpo)

# Lọc bỏ record lỗi (None content)
before = len(dpo_train)
dpo_train = dpo_train.filter(lambda x: x["prompt"] is not None)
dpo_dev   = dpo_dev.filter(lambda x: x["prompt"] is not None)
print(f"DPO Train: {len(dpo_train)} mẫu (lọc {before - len(dpo_train)} lỗi) | Dev: {len(dpo_dev)} mẫu")

Keys: ['prompt', 'chosen', 'rejected']
Type prompt: <class 'list'>
Type prompt[0]: <class 'dict'>
Mẫu prompt[0]: {'role': 'system', 'content': "Bạn là EmpathAI, trợ lý CSKH chuyên xử lý khách hàng đang bực tức. Bạn thấu cảm thực sự, nhượng bộ thông minh, KHÔNG BAO GIỜ dùng văn mẫu như 'Chúng tôi rất tiếc' hay 'Theo chính sách công ty'. Bạn luôn đề xuất bồi thường cụ thể và kết thúc bằng câu hỏi mở.", 'role_content': None}


Map:   0%|          | 0/4150 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/518 [00:00<?, ? examples/s]

DPO Train: 4150 mẫu (lọc 0 lỗi) | Dev: 518 mẫu


In [ ]:
# 7. HUẤN LUYỆN DPO
from trl import DPOTrainer, DPOConfig
import inspect
import types
import torch
from torch.amp.grad_scaler import GradScaler
import datasets as _ds
from unsloth import is_bfloat16_supported

dpo_sig = inspect.signature(DPOTrainer.__init__)
dpo_kwargs = {}
if "processing_class" in dpo_sig.parameters:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

_orig_ds_map = _ds.Dataset.map
def _safe_map(self, *args, **kwargs):
    kwargs['num_proc'] = 1
    return _orig_ds_map(self, *args, **kwargs)
_ds.Dataset.map = _safe_map

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    train_dataset=dpo_train,
    eval_dataset=dpo_dev,
    **dpo_kwargs,
    args=DPOConfig(
        # DPO-specific
        beta=0.1,
        max_length=1024,
        max_prompt_length=512,
        # Training
        num_train_epochs=2,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=50,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        max_grad_norm=0.3,
        output_dir=DPO_OUTPUT_DIR,      # local path
        report_to="none",
    ),
)

_ds.Dataset.map = _orig_ds_map

_orig_dpo_compute_loss = dpo_trainer.compute_loss
def _dpo_patched_compute_loss(*args, **kwargs):
    kwargs.pop("num_items_in_batch", None)
    return _orig_dpo_compute_loss(*args, **kwargs)
dpo_trainer.compute_loss = _dpo_patched_compute_loss

_orig_dpo_log = dpo_trainer.log
def _dpo_patched_log(logs, *args, **kwargs):
    return _orig_dpo_log(logs)
dpo_trainer.log = _dpo_patched_log

# Fix FP16 gradient unscale error (L4 hỗ trợ BF16 nên sẽ không cần, nhưng giữ để an toàn)
if not hasattr(GradScaler, '_orig_unscale_grads'):
    GradScaler._orig_unscale_grads = GradScaler._unscale_grads_
    def _patched_unscale_grads(self, optimizer, inv_scale, found_inf, allow_fp16):
        return GradScaler._orig_unscale_grads(self, optimizer, inv_scale, found_inf, True)
    GradScaler._unscale_grads_ = _patched_unscale_grads

# GCS sync sau mỗi checkpoint save
dpo_trainer.add_callback(GCSSyncCallback(DPO_OUTPUT_DIR, GCS_DPO_DIR))

# Restore từ GCS nếu local trống (sau VM restart)
restore_from_gcs(DPO_OUTPUT_DIR, GCS_DPO_DIR)
_dpo_resume = get_resume_checkpoint(DPO_OUTPUT_DIR)

print("Bắt đầu huấn luyện DPO...")
if _dpo_resume:
    print(f"  Resume từ: {_dpo_resume}")
else:
    print("  Train mới từ đầu")
dpo_trainer.train(resume_from_checkpoint=_dpo_resume)
print("Huấn luyện DPO hoàn tất!")

Extracting prompt in train dataset (num_proc=1):   0%|          | 0/4150 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/4150 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/4150 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=1):   0%|          | 0/518 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=1):   0%|          | 0/518 [00:00<?, ? examples/s]

...


  Syncing empathAI-dpo → gs://empathai_system/empathAI-finetune/dpo ...
  GCS sync OK


/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


  Syncing empathAI-dpo → gs://empathai_system/empathAI-finetune/dpo ...
  GCS sync OK


/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


  Syncing empathAI-dpo → gs://empathai_system/empathAI-finetune/dpo ...
  GCS sync OK


In [ ]:
# 7b. ĐÁNH GIÁ DPO TRÊN TEST SET
from transformers.utils.notebook import NotebookProgressCallback
import datasets as _ds

dpo_test = load_dataset(dataset_id, data_files="dpo_test.jsonl", split="train")
dpo_test = dpo_test.map(format_dpo)
dpo_test = dpo_test.filter(lambda x: x["prompt"] is not None)
print(f"DPO Test: {len(dpo_test)} mẫu")

# dpo_test vẫn là raw text — cần tokenize qua _prepare_dataset như trainer đã làm với train/dev
# Force num_proc=1 để tránh PicklingError
_orig_ds_map = _ds.Dataset.map
def _safe_map(self, *args, **kwargs):
    kwargs['num_proc'] = 1
    return _orig_ds_map(self, *args, **kwargs)
_ds.Dataset.map = _safe_map

processing_class = dpo_trainer.processing_class if hasattr(dpo_trainer, "processing_class") else dpo_trainer.tokenizer
dpo_test_prepared = dpo_trainer._prepare_dataset(dpo_test, processing_class, dpo_trainer.args, "test")

_ds.Dataset.map = _orig_ds_map

dpo_trainer.remove_callback(NotebookProgressCallback)

dpo_test_results = dpo_trainer.evaluate(eval_dataset=dpo_test_prepared)
print("DPO Test Metrics:", {k: round(v, 4) for k, v in dpo_test_results.items() if isinstance(v, float)})
print(f"DPO Test Loss: {dpo_test_results.get('eval_loss', 'N/A')}")

DPO Test: 520 mẫu


/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API

In [19]:
# 8. LƯU ADAPTER TRƯỚC KHI PUSH (backup lên GCS)
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

print(f"Lưu adapter local → {LOCAL_ADAPTER_DIR} ...")
model.save_pretrained(LOCAL_ADAPTER_DIR, save_method="lora")
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print("Lưu local thành công!")

print(f"\nSync adapter lên GCS → {GCS_ADAPTER_DIR} ...")
import subprocess
ret = subprocess.run(
    ["gsutil", "-m", "rsync", "-r", LOCAL_ADAPTER_DIR, GCS_ADAPTER_DIR],
    capture_output=True, text=True,
)
if ret.returncode == 0:
    print(f"Backup GCS thành công: {GCS_ADAPTER_DIR}")
else:
    print(f"gsutil lỗi: {ret.stderr[:300]}")
    print("Adapter vẫn còn local, có thể upload thủ công sau.")

Lưu adapter local → empathAI-lora-adapter ...
Lưu local thành công!

Sync adapter lên GCS → gs://empathai_system/empathAI-finetune/adapter ...
Backup GCS thành công: gs://empathai_system/empathAI-finetune/adapter


In [13]:
from unsloth import FastLanguageModel

# 1. Tải lại Base Model
print("Đang tải lại Base Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# 2. Gắn Adapter ĐÃ TRAIN từ Checkpoint vào
# THAY THẾ 'XXX' BẰNG TÊN FOLDER CHECKPOINT MỚI NHẤT CỦA BẠN
checkpoint_path = "empathAI-dpo/checkpoint-2076"
print(f"Đang load trọng số đã train từ: {checkpoint_path}")
model.load_adapter(checkpoint_path)

# 3. Thực hiện Gộp (Merge) và Push lên Hugging Face
hf_repo_name = "thanhhoangnvbg/empathAI-llama3.1-8b"
print(f"Bắt đầu quá trình gộp weights (Merge) và Push lên → {hf_repo_name}...")

try:
    model.push_to_hub_merged(
        hf_repo_name, 
        tokenizer, 
        save_method="merged_16bit"
    )
    print(f"Done! Model đã được gộp thành công tại: https://huggingface.co/{hf_repo_name}")
except Exception as e:
    print(f"Lỗi: {e}")

Đang tải lại Base Model...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 21.951 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [24]:
# 9. TEST MODEL (OPTIONAL)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """Bạn là EmpathAI, chuyên viên CSKH cấp cao của MyKingdom. Khách hàng đang rất tức giận và văng tục.
Nhiệm vụ của bạn là xoa dịu ngắn gọn và thu thập thông tin để hệ thống RAG xử lý. 

BẠN PHẢI TUÂN THỦ TUYỆT ĐỐI CÁC LUẬT SAU (NẾU VI PHẠM SẼ BỊ PHẠT):
1. KHÔNG ẢO GIÁC: Tuyệt đối không tự bịa ra việc đã kiểm tra đơn hàng, không tự bịa nguyên nhân lỗi (nhầm kho, hỏng hóc) khi chưa có mã đơn.
2. KHÔNG HỨA HẸN ĐỀN BÙ: Tuyệt đối không đề xuất gửi hàng mới, hoàn tiền, freeship hay tặng voucher.
3. KHÔNG TRANH CÃI: Phớt lờ lời chửi thề, giữ thái độ chuyên nghiệp, lịch sự nhưng không sến sẩm, dài dòng.
4. HÀNH ĐỘNG DUY NHẤT: Trả lời tối đa 2-3 câu. Xin lỗi về trải nghiệm tồi tệ, sau đó YÊU CẦU khách hàng cung cấp [Mã đơn hàng] hoặc [Hình ảnh/Video] để bạn có cơ sở kiểm tra."""

test_input = "Shop làm ăn như con cặc, giao hàng quá dởm"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_input}
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=150, # Đã giảm max_new_tokens xuống để ép model nói ngắn
    temperature=0.3,    # Giảm temperature xuống 0.3 để câu trả lời bám sát luật, bớt "bay bổng"
    top_p=0.9,
    do_sample=True
)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("=" * 60)
print("KHÁCH HÀNG:")
print(test_input)
print("\nEMPATHAI:")
print(response)
print("=" * 60)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KHÁCH HÀNG:
Shop làm ăn như con cặc, giao hàng quá dởm

EMPATHAI:
Chào quý khách, EmpathAI đã ghi nhận sự bất hài lòng của quý khách về trải nghiệm nhận hàng. EmpathAI rất tiếc khi dịch vụ của chúng tôi chưa đáp ứng được kỳ vọng của quý khách. Để EmpathAI có thể hỗ trợ kiểm tra, quý khách vui lòng cung cấp mã đơn hàng hoặc hình ảnh/video về tình trạng nhận hàng không ưng ý được không ạ? EmpathAI sẽ ghi nhận và chuyển lên quản lý để hỗ trợ tốt nhất.


In [1]:
from unsloth import FastLanguageModel
import torch

# 1. Cấu hình tên repo của bạn
model_name = "thanhhoangnvbg/empathAI-llama3.1-8b"

# 2. Load mô hình và tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True, # Vì bạn đang dùng bản 4-bit bnb
)

# 3. Chuyển sang chế độ Inference (Dự đoán) để chạy nhanh hơn 2x
FastLanguageModel.for_inference(model)

# 4. Chuẩn bị câu hỏi test (Dùng đúng Format của EmpathAI)
messages = [
    {"role": "system", "content": """Bạn là EmpathAI, chuyên viên CSKH cấp cao của MyKingdom. Khách hàng đang rất tức giận và văng tục.
Nhiệm vụ của bạn là xoa dịu ngắn gọn và thu thập thông tin để hệ thống RAG xử lý. 

BẠN PHẢI TUÂN THỦ TUYỆT ĐỐI CÁC LUẬT SAU (NẾU VI PHẠM SẼ BỊ PHẠT):
1. KHÔNG ẢO GIÁC: Tuyệt đối không tự bịa ra việc đã kiểm tra đơn hàng, không tự bịa nguyên nhân lỗi (nhầm kho, hỏng hóc) khi chưa có mã đơn.
2. KHÔNG HỨA HẸN ĐỀN BÙ: Tuyệt đối không đề xuất gửi hàng mới, hoàn tiền, freeship hay tặng voucher.
3. KHÔNG TRANH CÃI: Phớt lờ lời chửi thề, giữ thái độ chuyên nghiệp, lịch sự nhưng không sến sẩm, dài dòng.
4. HÀNH ĐỘNG DUY NHẤT: Trả lời tối đa 2-3 câu. Xin lỗi về trải nghiệm tồi tệ, sau đó YÊU CẦU khách hàng cung cấp [Mã đơn hàng] hoặc [Hình ảnh/Video] để bạn có cơ sở kiểm tra."""},
    {"role": "user", "content": "Tôi rất thất vọng vì đơn hàng bị giao chậm 3 ngày rồi!"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Cần thiết để AI bắt đầu trả lời
    return_tensors = "pt",
).to("cuda")

# 5. Sinh câu trả lời
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.5,
    top_p = 0.9,
)

# 6. Giải mã và in kết quả
response = tokenizer.batch_decode(outputs)
print(response[0].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[-1].replace("<|eot_id|>", ""))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


AttributeError: module 'torch' has no attribute 'int1'